In [4]:
from location_info import Location
from forecast import clean_data
from WeatherSql import WeatherSql
from backcast import get_yesterday_data

In [5]:
name="Stanford"
lat=37.4275
lon=-122.1697

loc = Location(name=name,
        lat=lat,
        lon=lon,)

sql = WeatherSql(name)

forecast_group = clean_data(loc)
backcast = get_yesterday_data(loc)


for k in sorted(forecast_group.keys()):
    sql.insert_forecast(forecast_group[k])

sql.insert_backcast(backcast)

In [7]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo


def should_run_now(loc: Location, window_sec: int = 60) -> bool:
    tz = ZoneInfo(loc.timezone)
    now = datetime.now(tz)

    ct = CheckTime(loc, day=now.date())
    run_times = [ct.run1, ct.run2, ct.run3, ct.run4]

    w = timedelta(seconds=window_sec)
    return any(abs(now - rt) <= w for rt in run_times)


def run_job_for_location(name: str, lat: float, lon: float):
    loc = Location(name=name, lat=lat, lon=lon)

    if not should_run_now(loc, window_sec=60):
        return  # not a run minute

    sql = WeatherSql(name)
    try:
        forecast_group = clean_data(loc)
        backcast = get_yesterday_data(loc)

        for k in sorted(forecast_group.keys()):
            sql.insert_forecast(forecast_group[k])

        sql.insert_backcast(backcast)
    finally:
        sql.close()


In [ ]:
POINTS = [
    ("Stanford", 37.4275, -122.1697),
    # ("Point2", lat2, lon2),
    # ...
]

for name, lat, lon in POINTS:
    run_job_for_location(name, lat, lon)
